### Grid search to choose best params for the protein search

In [1]:
### Small grid search to choose the best params for the proteins search

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import xgboost as xgb
import matplotlib.patches as mpatches
from matplotlib.font_manager import FontProperties
import itertools
from sklearn.model_selection import train_test_split
%config Completer.use_jedi = False
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.metrics import plot_roc_curve
from sklearn.metrics import auc
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import GridSearchCV

In [2]:
clin = pd.read_csv('/Users/smasarone/Desktop/Atom_Trauma/Trauma_ML/XGB_grid_search/clin_ready.csv', header = 0, index_col=0)
prot = pd.read_csv('/Users/smasarone/Desktop/Trauma_data_scripts/data_final/data_last_modifications/raw_data_proteins_filtered.csv',header = 0, index_col = 0)
prot_list = pd.read_csv('/Users/smasarone/Desktop/prot_filtered.csv', header = 0, index_col=0)

prot = prot.filter(prot_list.iloc[:,0], axis = 1)
prot = prot.drop(labels = [1578], axis = 0)
clin = clin.drop(labels = [1578], axis = 0)
print(len(prot_list))
prot.shape

4979


(414, 4979)

In [3]:
y = clin['iss']

# split in train and test and then further split test in 2 (final result will be 70, 15, 15)
X_train, X_test_large, y_train, y_test_large = train_test_split(prot, y, shuffle = True, test_size = 0.3, random_state = 132)   #132     
X_test, X_val, y_test, y_val = train_test_split(X_test_large, y_test_large, shuffle = True, test_size = 0.5, random_state = 132)        

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("X_val:", X_val.shape)

X_train: (289, 4979)
X_test: (62, 4979)
X_val: (63, 4979)


In [4]:
#concatenate train and test 
train_test = X_train.append(X_test)
assert len(train_test) == (len(X_train) + len(X_test)), "Shape error - X"

y_train_test = y_train.append(y_test)
assert len(y_train_test) == (len(y_train) + len(y_test)), "Shape error - y"

In [5]:
depths = [1, 2, 4]
etas = [0.01, 0.2, 0.5]
y_train_test_ = [1 if i > 25 else 0 for i in y_train_test]

dtrain= xgb.DMatrix(train_test, label= y_train_test_)


for max_depth in depths:
    for eta in etas:
        
        params = {"eta":eta, 
                 "max_depth": max_depth}
        
        cv_results = xgb.cv(
            params,
            dtrain,
            num_boost_round=30,
            seed=42,
            nfold=3,
            metrics={'auc'},
            early_stopping_rounds=25
        )
        
        #print(cv_results)
        best_rounds = np.argmax(cv_results['test-auc-mean'])
        best_test = cv_results['test-auc-mean'][best_rounds]
        best_round_train = np.argmax(cv_results['train-auc-mean'])
        best_train = cv_results['train-auc-mean'][best_round_train] 
        current_depth = max_depth
        current_eta = eta
        print("best_result", best_test, "best_round:",best_rounds,"best training:",
              best_train, "depth:", current_depth, "best eta:", current_eta)
                                    

best_result 0.8489489999999998 best_round: 23 best training: 0.8986976666666666 depth: 1 best eta: 0.01
best_result 0.8678020000000001 best_round: 14 best training: 0.9864356666666666 depth: 1 best eta: 0.2
best_result 0.8617469999999999 best_round: 5 best training: 0.999774 depth: 1 best eta: 0.5
best_result 0.818976 best_round: 21 best training: 0.966778 depth: 2 best eta: 0.01
best_result 0.840468 best_round: 10 best training: 1.0 depth: 2 best eta: 0.2
best_result 0.8334830000000001 best_round: 12 best training: 1.0 depth: 2 best eta: 0.5
best_result 0.7761296666666667 best_round: 16 best training: 0.999338 depth: 4 best eta: 0.01
best_result 0.8116576666666667 best_round: 20 best training: 1.0 depth: 4 best eta: 0.2
best_result 0.8007153333333333 best_round: 4 best training: 1.0 depth: 4 best eta: 0.5


In [7]:
# test on the val set the performance
y_val_=[1 if i >25 else 0 for i in y_val]

dtrain = xgb.DMatrix(train_test, label= y_train_test_)
dtest = xgb.DMatrix(X_val, label=y_val_)
param = {'max_depth': 1, 'eta': 0.2, 'objective': 'binary:logistic', 'scale_pos_weight':3}
param['eval_metric'] = 'auc'
num_round = 30 
evallist = [(dtest, 'eval'), (dtrain, 'train')]
progress = {}
bst = xgb.train(param, dtrain, num_round, evallist, evals_result = progress, verbose_eval=5, early_stopping_rounds=25)     

[0]	eval-auc:0.71241	train-auc:0.83171
[5]	eval-auc:0.74213	train-auc:0.91159
[10]	eval-auc:0.76748	train-auc:0.93173
[15]	eval-auc:0.75350	train-auc:0.94650
[20]	eval-auc:0.74650	train-auc:0.95684
[25]	eval-auc:0.74301	train-auc:0.96669
[29]	eval-auc:0.73776	train-auc:0.97244
